In [ ]:
from pathlib import Path
import sys
import subprocess
import time
import importlib.metadata
import nbformat
import pandas as pd
from IPython.display import display

In [ ]:
ROOT = Path.cwd()
TASKS = ['a_import','b_prepare','c_fit','d_simulate','e_diagnose','f_plot']
# recreate ignored task directories and input links on a fresh checkout
for task in TASKS:
    (ROOT/task/'output').mkdir(parents=True,exist_ok=True)
(ROOT/'a_import/input').mkdir(parents=True,exist_ok=True)
links = {f'{task}/input':f'../{previous}/output' for previous,task in zip(TASKS,TASKS[1:])}
links.update({'c_fit/output/data':'../input','d_simulate/output/fit':'../input','e_diagnose/output/simulation':'../input'})
for relative,target in links.items():
    path = ROOT/relative
    if not path.is_symlink():path.symlink_to(target,target_is_directory=True)
plan = [path for task in TASKS for path in sorted((ROOT/task/'src').glob('*.ipynb'))]
pd.DataFrame({'order':range(1,len(plan)+1),'notebook':[str(p.relative_to(ROOT)) for p in plan]})

In [ ]:
# isolated IPython process per notebook; sequential and stop-on-error
WORKER = r"""import os, sys, traceback
from pathlib import Path
import nbformat
from IPython.core.interactiveshell import InteractiveShell
from IPython.utils.capture import capture_output
path = Path(sys.argv[1]).resolve()
os.chdir(path.parent)
notebook = nbformat.read(path, as_version=4)
shell = InteractiveShell.instance()
import matplotlib
matplotlib.use('module://matplotlib_inline.backend_inline')
from matplotlib_inline.backend_inline import configure_inline_support
configure_inline_support(shell, 'module://matplotlib_inline.backend_inline')
for index, cell in enumerate(notebook.cells, 1):
    with capture_output() as captured:
        result = shell.run_cell(cell.source, store_history=False)
    cell.execution_count = index
    cell.outputs = []
    if captured.stdout:
        cell.outputs.append(nbformat.v4.new_output('stream', name='stdout', text=captured.stdout))
    if captured.stderr:
        cell.outputs.append(nbformat.v4.new_output('stream', name='stderr', text=captured.stderr))
    for output in captured.outputs:
        cell.outputs.append(nbformat.v4.new_output('display_data', data=output.data, metadata=output.metadata))
    if result.error_before_exec or result.error_in_exec:
        error = result.error_before_exec or result.error_in_exec
        cell.outputs.append(nbformat.v4.new_output('error', ename=type(error).__name__, evalue=str(error),
                                                  traceback=traceback.format_exception(type(error), error, error.__traceback__)))
        nbformat.write(notebook, path)
        raise error
nbformat.write(notebook, path)
"""
runs = []
for path in plan:
    start = time.perf_counter()
    result = subprocess.run([sys.executable, '-c', WORKER, str(path)], capture_output=True, text=True)
    if result.returncode:
        raise RuntimeError(f'{path.relative_to(ROOT)}\n{result.stdout}\n{result.stderr}')
    runs.append({'notebook':str(path.relative_to(ROOT)), 'seconds':time.perf_counter()-start, 'status':'passed'})
    pd.DataFrame(runs).to_csv(ROOT/'pipeline_run.csv',index=False)
runs = pd.DataFrame(runs)
runs.to_csv(ROOT/'pipeline_run.csv',index=False)
display(runs)

In [ ]:
packages = ['numpy','pandas','scipy','scikit-learn','matplotlib','seaborn','nbformat','ipykernel']
versions = pd.DataFrame([{'package':p,'version':importlib.metadata.version(p)} for p in packages])
versions.to_csv(ROOT/'environment_versions.csv',index=False)
display(versions)